In [60]:
import morph_kgc
import json
import rdflib
import configparser

config = configparser.ConfigParser()
config.read("config.ini")


INPUT_FILENAME = config.get('config', 'trace_json')
OUTPUT_FILENAME = config.get('config', 'trace_ttl')
OCED_ONTOLOGY = config.get('config', 'oced_ontology')
RML_MAPPING = config.get('config', 'rml_mapping')

MOD_INPUT_FILENAME = INPUT_FILENAME.replace(".json", "-mod.json")


In [61]:

# Load the JSON data from a file
with open(INPUT_FILENAME, 'r') as file:
    data = json.load(file)

# Iterate over each event in the JSON data
domain_properties = {}
for event in data['events']:
    event_id = event['id']
    # Iterate over each attribute in the event
    for attribute in event['attributes']:
        attribute['id'] = event_id
        name = attribute['value_of']
        attribute['value_of'] = attribute['value_of'].replace(':', '_')
        domain_properties[name] = attribute['value_of']

# Iterate over each object in the JSON data
for object in data['objects']:
    object_id = object['id']
    # Iterate over each attribute in the event
    for attribute in object['attributes']:
        attribute['id'] = object_id

data['objects'] = [x for x in data['objects'] if x['instance_of_O'] not in ( "lifecycle:transition", "time:timestamp", "concept:name" )]

# Save the modified JSON data back to the file
with open(MOD_INPUT_FILENAME, 'w') as file:
    json.dump(data, file, indent=4)

print("JSON file updated successfully!")

    


JSON file updated successfully!


In [62]:
config_rml = """
            [DataSource1]
            mappings: """+RML_MAPPING+"""
            file_path: """+MOD_INPUT_FILENAME+"""
        """
graph = morph_kgc.materialize(config_rml)

ocedo = rdflib.Namespace('https://w3id.org/ocedo/core#')
aux = rdflib.Namespace('https://w3id.org/ocedo/aux#')
ocedr = rdflib.Namespace('https://w3id.org/ocedo/resource/')

graph.bind('oced', ocedo)
graph.bind('aux', aux)
graph.bind('res', ocedr)

graph.serialize(destination=OUTPUT_FILENAME, format='turtle')


INFO | 2025-03-19 03:11:25,696 | Parallelization is not supported for darwin when running as a library. If you need to speed up your data integration pipeline, please run through the command line.
INFO | 2025-03-19 03:11:25,992 | 25 mapping rules retrieved.
INFO | 2025-03-19 03:11:25,998 | Mapping partition with 25 groups generated.
INFO | 2025-03-19 03:11:25,999 | Maximum number of rules within mapping group: 1.
INFO | 2025-03-19 03:11:25,999 | Mappings processed in 0.301 seconds.
INFO | 2025-03-19 03:11:26,116 | Number of triples generated in total: 3138.


<Graph identifier=N67c6780120644cd094f94d1a50e00f4e (<class 'rdflib.graph.Graph'>)>

In [63]:
from rdflib import Graph
import requests

# Configuration
GRAPHDB_ENDPOINT = config.get('config', 'sparql_endpoint') + "/statements"
USERNAME = config.get('config', 'username')
PASSWORD = config.get('config', 'password')




In [64]:
from SPARQLWrapper import SPARQLWrapper, POST, JSON

# SPARQL Update query to delete all triples
delete_query = """
DELETE WHERE { 
  ?s ?p ?o 
}
"""

# Set up the SPARQLWrapper instance
sparql = SPARQLWrapper(GRAPHDB_ENDPOINT)

# If authentication is needed
sparql.setCredentials(USERNAME, PASSWORD)

# Configure the request
sparql.setQuery(delete_query)
sparql.setMethod(POST)
sparql.setReturnFormat(JSON)

# Execute the query
try:
    response = sparql.query()
    print("Repository cleared successfully.")
    print("Response:", response.response.read().decode())
except Exception as e:
    print(f"Failed to clear the repository: {e}")

Repository cleared successfully.
Response: 


In [65]:
# Load the TTL file into an RDFLib graph
g = Graph()
g.parse(OUTPUT_FILENAME, format="ttl")
g.parse(OCED_ONTOLOGY, format="ttl")
print(f"Graph has {len(g)} triples.")

# Serialize the graph to a string
data = g.serialize(format="turtle")

# Set headers for the request
headers = {
    "Content-Type": "application/x-turtle",
    "Accept": "application/json",
}

# Send the data to GraphDB
response = requests.post(
    GRAPHDB_ENDPOINT,
    data=data,
    headers=headers,
    auth=(USERNAME, PASSWORD)  # Remove if authentication is not enabled
)

# Check response status
if response.status_code == 204:  # 204 No Content is a successful response
    print("Data uploaded successfully to GraphDB.")
else:
    print(f"Failed to upload data. Status code: {response.status_code}")
    print(response.text)

Graph has 3249 triples.
Data uploaded successfully to GraphDB.
